# Notebook 02 — Data Cleaning
**Project:** Ola Bengaluru Rides — Cancellation Intelligence & Revenue Recovery  
**Dataset:** Bengaluru Ola Rides, January 2024  
**Author:** Joshit  
**Institute:** Newton School of Technology — DVA Capstone 2

Picking up from the extraction notebook. The raw file is never touched — all changes happen here and the cleaned output goes to `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('ready.')

In [ ]:
# update path if running locally: '../data/raw/Bengaluru_Ola.csv'
RAW_PATH = '/content/Bengaluru Ola (1).csv'

df = pd.read_csv(RAW_PATH)
original_shape = df.shape
print(f'loaded: {df.shape[0]} rows, {df.shape[1]} columns')

### Fixing column names
Renaming everything to snake_case — makes it much easier to work with in code. Also fixes the double space issue in `Cancelled  by Customer`.

In [ ]:
print('before:', list(df.columns))

In [ ]:
df.columns = [
    'date', 'time', 'booking_id', 'booking_status', 'customer_id',
    'vehicle_type', 'pickup_location', 'drop_location',
    'avg_vtat', 'avg_ctat',
    'cancelled_by_customer', 'cancel_reason_customer',
    'cancelled_by_driver', 'cancel_reason_driver',
    'incomplete_ride', 'incomplete_reason',
    'booking_value', 'payment_method',
    'ride_distance', 'driver_rating', 'customer_rating'
]

print('after:', list(df.columns))

### Parsing date and time, adding useful features
Date and time are just strings right now. Converting date to datetime and pulling out hour, day of week and a time-of-day bucket — these will be useful for the analysis.

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
df['hour'] = pd.to_datetime(df['time'], format='%H:%M:%S').dt.hour
df['day_of_week'] = df['date'].dt.day_name()
df['day'] = df['date'].dt.day

def time_bucket(hour):
    if 5 <= hour < 9:    return 'Early Morning'
    elif 9 <= hour < 12: return 'Morning'
    elif 12 <= hour < 16: return 'Afternoon'
    elif 16 <= hour < 20: return 'Evening'
    elif 20 <= hour < 24: return 'Night'
    else:                 return 'Late Night'

df['time_of_day'] = df['hour'].apply(time_bucket)

df[['date', 'time', 'hour', 'day_of_week', 'time_of_day']].head()

### Stripping whitespace from string columns
Just making sure there are no hidden leading/trailing spaces that could mess up groupby later.

In [ ]:
str_cols = ['booking_status', 'vehicle_type', 'pickup_location', 'drop_location',
            'payment_method', 'cancel_reason_customer', 'cancel_reason_driver', 'incomplete_reason']

for col in str_cols:
    df[col] = df[col].str.strip()

print('done stripping whitespace')
print('vehicle types:', sorted(df['vehicle_type'].unique()))
print('booking statuses:', df['booking_status'].unique())

### Handling nulls

Two different situations here:

1. **Structural nulls** — `booking_value`, `ride_distance`, `avg_vtat`, `avg_ctat`, `driver_rating`, `customer_rating`, `payment_method` are all null for non-successful rides. That's expected — a cancelled ride has no fare. Leaving these as NaN.

2. **Reason columns** — filling with `'Not Applicable'` since null here just means that cancellation type didn't happen.

In [ ]:
df['cancel_reason_customer'] = df['cancel_reason_customer'].fillna('Not Applicable')
df['cancel_reason_driver'] = df['cancel_reason_driver'].fillna('Not Applicable')
df['incomplete_reason'] = df['incomplete_reason'].fillna('Not Applicable')

print('reason column nulls after fill:')
print(df[['cancel_reason_customer', 'cancel_reason_driver', 'incomplete_reason']].isnull().sum())

In [ ]:
# verifying structural nulls match non-successful ride count
non_success = (df['booking_status'] != 'Success').sum()
bv_nulls = df['booking_value'].isnull().sum()

print(f'non-successful rides : {non_success:,}')
print(f'booking_value nulls  : {bv_nulls:,}')
print(f'match: {non_success == bv_nulls}')

### Fixing data types
The flag columns (cancelled_by_customer, cancelled_by_driver, incomplete_ride) should be integers not floats.

In [ ]:
df['cancelled_by_customer'] = df['cancelled_by_customer'].astype(int)
df['cancelled_by_driver'] = df['cancelled_by_driver'].astype(int)
df['incomplete_ride'] = df['incomplete_ride'].astype(int)

print(df[['cancelled_by_customer', 'cancelled_by_driver', 'incomplete_ride']].dtypes)

### Dealing with duplicate booking IDs
Found 133 duplicate booking IDs in the extraction notebook. Checking if they are truly the same row or different data with the same ID.

In [ ]:
dupes = df[df['booking_id'].duplicated(keep=False)].sort_values('booking_id')
print(f'rows with duplicate booking IDs: {len(dupes)}')
dupes.head(10)

In [ ]:
# dropping full duplicate rows, keeping first occurrence
before = len(df)
df = df.drop_duplicates(subset='booking_id', keep='first')
after = len(df)

print(f'rows before: {before:,}')
print(f'rows after : {after:,}')
print(f'dropped    : {before - after}')

### Adding derived columns
A few extra columns that will make the analysis much cleaner — flags, cancellation party, revenue lost estimate and area number.

In [ ]:
df['is_successful'] = (df['booking_status'] == 'Success').astype(int)
df['is_cancelled'] = df['booking_status'].isin(['Cancelled by Driver', 'Cancelled by Customer']).astype(int)

def cancel_party(row):
    if row['booking_status'] == 'Cancelled by Driver':   return 'Driver'
    elif row['booking_status'] == 'Cancelled by Customer': return 'Customer'
    elif row['booking_status'] == 'Incomplete':           return 'Incomplete'
    else:                                                  return 'None'

df['cancellation_party'] = df.apply(cancel_party, axis=1)

avg_fare = df['booking_value'].mean()
df['revenue_lost'] = df.apply(lambda r: round(avg_fare, 2) if r['is_successful'] == 0 else 0, axis=1)

df['pickup_area_num'] = df['pickup_location'].str.extract(r'(\d+)').astype(float)

print(f'avg fare used for revenue_lost estimate: ₹{avg_fare:.2f}')
df[['booking_status', 'is_successful', 'is_cancelled', 'cancellation_party', 'revenue_lost']].head(8)

### Outlier detection
Using IQR to check for outliers in the numeric columns. Not removing anything — just flagging so we can filter later if needed.

In [ ]:
df_success = df[df['is_successful'] == 1].copy()
num_cols = ['booking_value', 'ride_distance', 'avg_vtat', 'avg_ctat', 'driver_rating', 'customer_rating']

print('outlier check (successful rides only):\n')
for col in num_cols:
    Q1 = df_success[col].quantile(0.25)
    Q3 = df_success[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n = ((df_success[col] < lower) | (df_success[col] > upper)).sum()
    print(f'{col:<20} lower: {lower:>8.2f}  upper: {upper:>8.2f}  outliers: {n}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df_success[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#AED6F1', color='#1A56A0'),
                    medianprops=dict(color='#E74C3C', linewidth=2))
    axes[i].set_title(col.replace('_', ' '), fontsize=11, fontweight='bold')

plt.suptitle('Outlier Check — Successful Rides', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/02_outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# flagging outliers in booking_value and ride_distance
for col in ['booking_value', 'ride_distance']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    flag_col = f'{col}_outlier_flag'
    df[flag_col] = ((df[col] < lower) | (df[col] > upper)).astype(float)
    df.loc[df[col].isnull(), flag_col] = np.nan
    print(f'{col}: {int(df[flag_col].sum())} rows flagged')

### Validation — checking everything looks right before saving

In [ ]:
print(f'original shape : {original_shape}')
print(f'cleaned shape  : {df.shape}')
print(f'rows removed   : {original_shape[0] - df.shape[0]}')
print(f'columns added  : {df.shape[1] - original_shape[1]}')
print()
print('remaining nulls:')
remaining = df.isnull().sum()
print(remaining[remaining > 0])
print()
print('booking status counts:')
print(df['booking_status'].value_counts())

In [ ]:
party_counts = df['cancellation_party'].value_counts()

plt.figure(figsize=(8, 5))
colors = ['#2ECC71', '#E74C3C', '#E67E22', '#F39C12']
bars = plt.bar(party_counts.index, party_counts.values, color=colors, edgecolor='white')
plt.title('Ride Outcome by Cancellation Party', fontsize=13, fontweight='bold')
plt.ylabel('Number of Rides')
for bar, val in zip(bars, party_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
             f'{val:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('/content/02_cancellation_party.png', dpi=150, bbox_inches='tight')
plt.show()

### Saving cleaned dataset

In [ ]:
CLEAN_PATH = '/content/ola_bengaluru_cleaned.csv'
df.to_csv(CLEAN_PATH, index=False)

print(f'saved to: {CLEAN_PATH}')
print(f'final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('proceed to 03_eda.ipynb')

### What was done in this notebook

- Renamed all columns to snake_case (fixed double space issue too)
- Parsed date to datetime, extracted hour, day_of_week, time_of_day
- Stripped whitespace from all string columns
- Left structural nulls as NaN — they belong to non-successful rides
- Filled cancellation reason nulls with 'Not Applicable'
- Fixed data types for flag columns
- Investigated and dropped 133 duplicate booking IDs
- Added derived columns: is_successful, is_cancelled, cancellation_party, revenue_lost, pickup_area_num
- Flagged outliers in booking_value and ride_distance (not removed)
- Saved cleaned file to data/processed/